# Feature engineering for shelter occupancy prediction

This notebook builds on the exploratory analysis and prepares features
for the regression models that will forecast daily shelter occupancy
rates. We create temporal indicators, rolling statistics, lag features,
and encode the shelter type category.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import fetch_data, preprocess, add_rolling_features
from src.model import encode_categorical, DEFAULT_FEATURES

pd.set_option('display.max_columns', 50)

## 1. Load and preprocess the raw data

In [ ]:
raw_df = fetch_data(use_cache=True)
df = preprocess(raw_df)
print(f"Preprocessed shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Columns: {list(df.columns)}")

## 2. Temporal features

The `preprocess` function already extracts `day_of_week`, `month`, `year`,
and `day_of_month` from the date column. Let us verify these and look at
their distributions.

In [ ]:
temporal_cols = ['day_of_week', 'month', 'year', 'day_of_month']
print(df[temporal_cols].describe())
print(f"\nDay-of-week value counts (0=Mon, 6=Sun):")
print(df['day_of_week'].value_counts().sort_index())

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Day of week', 'Month', 'Year', 'Day of month'))

for i, col in enumerate(temporal_cols):
    row, col_idx = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=df[col], name=col, nbinsx=len(df[col].unique())),
        row=row + 1, col=col_idx + 1
    )

fig.update_layout(height=500, title_text='Temporal feature distributions',
                  showlegend=False)
fig.show()

## 3. Rolling statistics (7-day and 30-day)

Rolling averages smooth out daily noise and capture recent occupancy
trends per shelter. We compute them grouped by shelter to prevent
information leaking across different locations.

In [ ]:
df = add_rolling_features(df)
print("Rolling and lag columns added:")
print([c for c in df.columns if 'rolling' in c or 'lag' in c])
print(f"\nShape after feature engineering: {df.shape}")
df[['date', 'shelter', 'occupancy_rate', 'rolling_7d_occupancy',
    'rolling_30d_occupancy', 'lag_1d_occupancy', 'lag_7d_occupancy']].head(15)

In [ ]:
# Show the rolling features for a single shelter to verify correctness
sample_shelter = df['shelter'].dropna().unique()[0]
shelter_df = df[df['shelter'] == sample_shelter].sort_values('date')

fig = go.Figure()
fig.add_trace(go.Scatter(x=shelter_df['date'], y=shelter_df['occupancy_rate'],
                         name='Daily', mode='lines', opacity=0.4))
fig.add_trace(go.Scatter(x=shelter_df['date'], y=shelter_df['rolling_7d_occupancy'],
                         name='7-day rolling avg', mode='lines'))
fig.add_trace(go.Scatter(x=shelter_df['date'], y=shelter_df['rolling_30d_occupancy'],
                         name='30-day rolling avg', mode='lines'))
fig.update_layout(title=f'Rolling averages for {sample_shelter}',
                  xaxis_title='Date', yaxis_title='Occupancy rate',
                  height=400)
fig.show()

## 4. Lag features

Lag features let the model use past occupancy as a predictor.
`lag_1d_occupancy` is yesterday's rate, and `lag_7d_occupancy` is the
rate from the same day of the previous week. Both are computed per
shelter.

In [ ]:
# Check NaN counts introduced by lags
lag_cols = ['lag_1d_occupancy', 'lag_7d_occupancy']
print("NaN counts in lag features:")
print(df[lag_cols].isna().sum())
print(f"\nTotal rows: {len(df)}")
print(f"Rows with all lag features present: {df[lag_cols].dropna().shape[0]}")

In [ ]:
# Scatter: lag_1d vs current occupancy
sample = df.dropna(subset=['lag_1d_occupancy']).sample(n=min(5000, len(df)), random_state=42)

fig = px.scatter(
    sample, x='lag_1d_occupancy', y='occupancy_rate',
    opacity=0.3,
    title='Yesterday occupancy vs today occupancy',
    labels={'lag_1d_occupancy': 'Lag 1-day occupancy',
            'occupancy_rate': 'Current occupancy rate'},
    trendline='ols'
)
fig.update_layout(height=400)
fig.show()

## 5. Encode shelter type

Shelter type is a categorical variable. We use label encoding so tree-based
models can consume it directly. The `encode_categorical` function from
`src/model.py` handles this.

In [ ]:
df_enc, encoders = encode_categorical(df, columns=['sheltertype'])

print(f"Encoder classes for sheltertype: {list(encoders['sheltertype'].classes_)}")
print(f"\nEncoded column sample:")
df_enc[['sheltertype', 'sheltertype_encoded']].drop_duplicates().sort_values('sheltertype_encoded')

## 6. Feature distributions

Below we visualise the distributions of all features that will go into the
model to check for skew, outliers, and potential scaling issues.

In [ ]:
feature_cols = DEFAULT_FEATURES
print(f"Model features ({len(feature_cols)}): {feature_cols}")

available = [c for c in feature_cols if c in df_enc.columns]
feat_df = df_enc[available].dropna()
print(f"\nRows with all features present: {len(feat_df)}")
feat_df.describe()

In [ ]:
# Distribution of each feature
n_features = len(available)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=available)

for i, col in enumerate(available):
    row, col_idx = divmod(i, n_cols)
    fig.add_trace(
        go.Histogram(x=feat_df[col], name=col, nbinsx=40),
        row=row + 1, col=col_idx + 1
    )

fig.update_layout(height=200 * n_rows, title_text='Feature distributions',
                  showlegend=False)
fig.show()

## 7. Feature correlations

A correlation heatmap shows how strongly each feature relates to the
target (occupancy rate) and to other features. High intercorrelation
among rolling/lag features is expected.

In [ ]:
corr_cols = available + ['occupancy_rate']
corr_df = df_enc[corr_cols].dropna()
corr = corr_df.corr()

fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Feature correlation matrix (including target)',
    aspect='auto'
)
fig.update_layout(height=550, width=700)
fig.show()

In [ ]:
# Correlation of each feature with the target
target_corr = corr['occupancy_rate'].drop('occupancy_rate').sort_values(ascending=False)
print("Correlation with occupancy_rate:")
print(target_corr.to_string())

## 8. Summary

Feature engineering produced the following set of predictors:

| Feature group | Variables | Rationale |
|---|---|---|
| Temporal | day_of_week, month, year, day_of_month | Capture weekly and seasonal cycles |
| Capacity | capacity | Shelter size affects occupancy dynamics |
| Rolling averages | rolling_7d, rolling_30d | Smooth recent trend per shelter |
| Lag values | lag_1d, lag_7d | Autocorrelation in occupancy series |
| Categorical | sheltertype_encoded | Different shelter categories have different baselines |

The lag and rolling features show the strongest correlation with the target.
Rows with NaN lags (the first few days per shelter) will be dropped during
model training. The next notebook trains and compares regression models.